In [1]:
import torch, torch.nn as nn, torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import numpy as np

In [2]:
# ---- Hyperparameters ----
GAMMA, LAMBDA, EPSILON = 0.99, 0.95, 0.2
LR = 3e-3
ROLLOUT_STEPS = 500
UPDATES = 150

In [3]:
# ---- Actor-Critic network ----
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.body = nn.Sequential(nn.Linear(state_dim, 64),nn.Tanh())
        self.actor = nn.Linear(64, action_dim)  # outputs Left/Right scores
        self.critic = nn.Linear(64, 1)          # outputs V(s)

    def forward(self, x):
        x = self.body(x)
        return self.actor(x), self.critic(x).squeeze(-1)

In [4]:
# ---- GAE: compute advantages by walking backward through the rollout ----
def compute_gae(rewards, values, next_value, dones):
    values = values + [next_value]
    advantages, gae = [0] * len(rewards), 0
    for t in reversed(range(len(rewards))):
        mask = 1 - dones[t]
        delta = (rewards[t] + GAMMA * values[t + 1] * mask - values[t])
        gae = delta + GAMMA * LAMBDA * mask * gae
        advantages[t] = gae
    returns = [a + v for a, v in zip(advantages, values[:-1])]
    return advantages, returns

In [5]:
# ---- Main training loop ----
def train():
    env = gym.make("CartPole-v1")
    model = ActorCritic(env.observation_space.shape[0], env.action_space.n)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    state, _ = env.reset()

    for update in range(UPDATES):
        states, actions, rewards, dones, old_log_probs, values = [], [], [], [], [], []

        # ---- 1. Collect experience by playing CartPole with current policy ----

        for _ in range(ROLLOUT_STEPS):
            s = torch.FloatTensor(state)
            logits, value = model(s)
            dist = Categorical(logits=logits)
            action = dist.sample()  # 0 = push Left, 1 = push Right
            
            next_state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated  # pole fell or episode timed out

            states.append(state);actions.append(action.item())
            rewards.append(reward);dones.append(done)
            old_log_probs.append(dist.log_prob(action).item())
            values.append(value.item())

            state = next_state if not done else env.reset()[0]

        with torch.no_grad():
            _, next_value = model(torch.FloatTensor(state))

        # ---- 2. Compute GAE advantages + returns ----
        advantages, returns = compute_gae(rewards, values, next_value.item(), dones)

        states_t = torch.FloatTensor(np.array(states))
        actions_t = torch.LongTensor(actions)
        old_log_probs_t = torch.FloatTensor(old_log_probs)
        returns_t = torch.FloatTensor(returns)
        advantages_t = torch.FloatTensor(advantages)

        advantages_t = ((advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8))

        # ---- 3. PPO update ----
        logits, values_pred = model(states_t)
        dist = Categorical(logits=logits)
        new_log_probs = dist.log_prob(actions_t)
        entropy = dist.entropy().mean()

        # Clipped policy (surrogate) loss
        ratio = torch.exp(new_log_probs - old_log_probs_t)
        surr1 = ratio * advantages_t
        surr2 = (torch.clamp(ratio,1 - EPSILON,1 + EPSILON)* advantages_t)
        policy_loss = -torch.min(surr1,surr2).mean()

        # Clipped value loss
        value_clipped = returns_t + torch.clamp(values_pred - returns_t,-EPSILON,EPSILON)
        value_loss = 0.5 * torch.max(
            (values_pred - returns_t) ** 2,
            (value_clipped - returns_t) ** 2
        ).mean()

        # Total loss = policy loss + value loss - entropy bonus
        loss = (policy_loss+ 0.5 * value_loss- 0.01 * entropy)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if update % 10 == 0:
            print(f"Update {update} | "f"avg reward: {sum(rewards) / max(1, sum(dones)):.1f} | "f"entropy: {entropy.item():.3f}")

    env.close()

if __name__ == "__main__":
    train()

Update 0 | avg reward: 25.0 | entropy: 0.691
Update 10 | avg reward: 33.3 | entropy: 0.662
Update 20 | avg reward: 55.6 | entropy: 0.645
Update 30 | avg reward: 55.6 | entropy: 0.621
Update 40 | avg reward: 83.3 | entropy: 0.629
Update 50 | avg reward: 71.4 | entropy: 0.616
Update 60 | avg reward: 100.0 | entropy: 0.623
Update 70 | avg reward: 125.0 | entropy: 0.610
Update 80 | avg reward: 83.3 | entropy: 0.609
Update 90 | avg reward: 100.0 | entropy: 0.612
Update 100 | avg reward: 125.0 | entropy: 0.620
Update 110 | avg reward: 83.3 | entropy: 0.621
Update 120 | avg reward: 100.0 | entropy: 0.640
Update 130 | avg reward: 71.4 | entropy: 0.607
Update 140 | avg reward: 71.4 | entropy: 0.633
